## Importing required Modules and Libraries

In [63]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
import pandas as pd 
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import xgboost as xgb
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

## Importing Data:

In [64]:
path = '/kaggle/input/competitions/playground-series-s6e4/'

train_df = pd.read_csv(path + 'train.csv')
display(train_df.head())
print('Training set: ', train_df.shape)

test_df = pd.read_csv(path + 'test.csv')
display(test_df.head())
print('Test set: ', test_df.shape)

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


Training set:  (630000, 21)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
0,630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
1,630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
2,630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
3,630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
4,630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central


Test set:  (270000, 20)


In [65]:
target = 'Irrigation_Need'

X = train_df.drop(columns=['id', target], axis=1)
y = train_df[target].map({'Low': 0, 'Medium': 1, 'High': 2})

In [66]:
num_cols = X.select_dtypes(include='number').columns
cat_cols = X.select_dtypes("object").columns

In [67]:
X[cat_cols] = X[cat_cols].astype('category')
test_df[cat_cols] = test_df[cat_cols].astype('category')

In [68]:
X_test = test_df.drop('id', axis=1)

## OOF predictions without scaling and OHE:

In [69]:
xgb_params = {
    'learning_rate': 0.03, 
    'max_depth': 4, 
    'min_child_weight': 2,
    'subsample': 0.98,
    'gamma': 4.2, 
    'colsample_bytree': 0.5, 
    'lambda': 0.001,
    'alpha': 4.1,
    'n_estimators': 50_000, 
    'objective': 'multi:softprob',
    'eval_metric': 'mlogloss',
    'device': 'cuda',
    'enable_categorical': True,
}

In [70]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
bas_scores = []
n_classes = y.nunique()
oof_test = np.zeros((len(X_test), n_classes))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    sw = compute_sample_weight(class_weight="balanced", y=y_tr) #used when the dataset is imbalanced, minority clas gets more importance
    
    model = XGBClassifier(
        **xgb_params,
        random_state=42,
        
        callbacks=[
            xgb.callback.EarlyStopping( #early stopping callback
                rounds=200,
                save_best=True,
            )
        ]
    )

    model.fit(
        X_tr,
        y_tr,
        sample_weight=sw,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    preds = model.predict(X_val)

    test_preds = model.predict_proba(X_test)
    oof_test += test_preds / skf.n_splits

    bas = balanced_accuracy_score(y_val, preds)
    bas_scores.append(bas)
    print(f"Fold {fold} BAS: {bas}")
print(f"Mean BAS: {np.mean(bas_scores)}")

Fold 0 BAS: 0.9709237028468536
Fold 1 BAS: 0.9726111663957253
Fold 2 BAS: 0.9728757329256101
Fold 3 BAS: 0.9719502922631239
Fold 4 BAS: 0.9709273918954079
Mean BAS: 0.9718576572653441


In [71]:
preds = oof_test.argmax(axis=1)
preds = pd.Series(preds).map({0: "Low", 1: "Medium", 2: "High"})

submission = pd.DataFrame({
    "id": test_df["id"],
    target: preds
})

submission.to_csv("submission_without_scaling.csv", index=False)

## With scaling and OHE(one hot encoding):
Wont be doing it in the cleanest way possible in a little time crunch, so overlook it please

In [72]:
#redefining X and y
X = train_df.drop(columns=['id', target], axis=1)
y = train_df[target].map({'Low': 0, 'Medium': 1, 'High': 2})

X_test = pd.read_csv(path + 'test.csv').drop('id', axis=1)

In [73]:
num_cols = X.select_dtypes(include=['number']).columns
cat_cols = X.select_dtypes(include=['object']).columns

In [74]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)
    ]
)

In [75]:
X_processed = preprocessor.fit_transform(X)
X_test_processed = preprocessor.transform(X_test)

In [76]:
xgb_params = {
    'learning_rate': 0.03, 
    'max_depth': 4, 
    'min_child_weight': 2,
    'subsample': 0.98,
    'gamma': 4.2, 
    'colsample_bytree': 0.5, 
    'lambda': 0.001,
    'alpha': 4.1,
    'n_estimators': 50_000, 
    'objective': 'multi:softprob',
    'eval_metric': 'mlogloss',
    'device': 'cuda',
}

In [77]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
bas_scores = []
n_classes = y.nunique()
oof_test = np.zeros((len(X_test_processed), n_classes))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_processed, y)):
    X_tr, X_val = X_processed[tr_idx], X_processed[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    sw = compute_sample_weight(class_weight="balanced", y=y_tr)
    
    model = XGBClassifier(
        **xgb_params,
        random_state=42,
        
        callbacks=[
            xgb.callback.EarlyStopping( #early stopping callback
                rounds=200,
                save_best=True,
            )
        ]
    )

    model.fit(
        X_tr,
        y_tr,
        sample_weight=sw,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    preds = model.predict(X_val)

    test_preds = model.predict_proba(X_test_processed)
    oof_test += test_preds / skf.n_splits

    bas = balanced_accuracy_score(y_val, preds)
    bas_scores.append(bas)
    print(f"Fold {fold} BAS: {bas}")
print(f"Mean BAS: {np.mean(bas_scores)}")

Fold 0 BAS: 0.970161674720075
Fold 1 BAS: 0.9730451261924391
Fold 2 BAS: 0.972525697179598
Fold 3 BAS: 0.9718029374193579
Fold 4 BAS: 0.9713169518496912
Mean BAS: 0.9717704774722323


In [78]:
preds = oof_test.argmax(axis=1)
preds = pd.Series(preds).map({0: "Low", 1: "Medium", 2: "High"})

submission = pd.DataFrame({
    "id": test_df["id"],
    target: preds
})

submission.to_csv("submission_with_scaling.csv", index=False)